# v1 - CSV >> parquet + load check

In [ ]:
import glob
import os

import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "src", "config.py")):
    parent = os.path.dirname(ROOT)
    assert parent != ROOT, "repo root not found above cwd"
    ROOT = parent
REAL = os.path.join(ROOT, "src", "data", "real")

# detection/shap/ is listed explicitly: 00_SHAP.ipynb falls back to CSV when env-v1's pandas
# cannot write parquet, and its attributions land one level down from detection/.
FILES = sorted(
    os.path.relpath(p, REAL)
    for p in glob.glob(os.path.join(REAL, "inputs", "*.csv"))
    + glob.glob(os.path.join(REAL, "detection", "*.csv"))
    + glob.glob(os.path.join(REAL, "detection", "shap", "*.csv"))
)
print("repo root:", ROOT)
print(len(FILES), "csv files:", FILES)


In [ ]:
def csv_to_parquet(src, dst):
    n = 0
    writer = None
    with pacsv.open_csv(src) as reader:
        for batch in reader:
            if writer is None:
                writer = pq.ParquetWriter(dst, batch.schema, compression="zstd")
            writer.write_batch(batch)
            n += batch.num_rows
    writer.close()
    return n


for rel in FILES:
    src = os.path.join(REAL, rel)
    dst = src[:-4] + ".parquet"
    n_csv = csv_to_parquet(src, dst)
    n_pq = pq.read_metadata(dst).num_rows
    mb = lambda p: os.path.getsize(p) / 1024**2
    print(f"{rel:<28} rows csv {n_csv:>9} == parquet {n_pq:>9}  "
          f"{mb(src):>9.1f} MB -> {mb(dst):>7.1f} MB")
    assert n_csv == n_pq, rel


In [ ]:
for rel in FILES:
    dst = os.path.join(REAL, rel[:-4] + ".parquet")
    df = pd.read_parquet(dst)
    print("=" * 70)
    print(rel[:-4] + ".parquet", " shape", df.shape)
    print(df.dtypes.to_string())
    print(df.head(3).to_string())
